# Segmentação de Água em Imagens de Satélite — Exploração e Tentativas

**Competição:** 2026 ITU · Ingenuity Cup AI and Space Computing Challenge  
**Dataset:** GID (Gaofen Image Dataset) — imagens de satélite de alta resolução  
**Autor:** Luiz Felipe Castro — luiz.castro@usp.br  

---

Este notebook registra as tentativas iniciais de desenvolvimento do modelo, incluindo erros cometidos e decisões que foram descartadas. O pipeline final e limpo está no notebook `02_pipeline_final.ipynb`.

## O problema

O dataset GID contém imagens RGB de satélite (sensor Gaofen-2) e máscaras de segmentação com múltiplas classes de uso do solo. O objetivo da competição era identificar especificamente a **classe água** nessas imagens.

As máscaras originais são imagens em escala de cinza onde cada valor de pixel representa uma classe. O primeiro desafio foi identificar qual valor correspondia à água — isso não estava explícito na documentação e exigiu inspeção visual dos dados.


## 1. Inspeção inicial dos dados

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
from glob import glob

# Caminhos locais (ajuste conforme seu ambiente)
LBL_DIR = r'data/Train/GID-label'
IMG_DIR  = r'data/Train/GID-img-1'

# Inspeciona os valores únicos presentes nas máscaras
# para descobrir quais classes existem no dataset
label_files = glob(os.path.join(LBL_DIR, '*.png'))[:20]
valores_unicos = set()
for f in label_files:
    m = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
    valores_unicos.update(np.unique(m).tolist())

print('Valores únicos encontrados nas máscaras:', sorted(valores_unicos))
print('Cada valor representa uma classe de uso do solo.')


In [ ]:
# Visualização de uma imagem e sua máscara correspondente
# para entender visualmente o que cada valor representa
sample_label = label_files[0]
sample_img   = os.path.join(IMG_DIR, os.path.basename(sample_label).replace('.png', '.jpg'))

img  = cv2.cvtColor(cv2.imread(sample_img), cv2.COLOR_BGR2RGB)
mask = cv2.imread(sample_label, cv2.IMREAD_GRAYSCALE)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(img)
axes[0].set_title('Imagem original (RGB)')
axes[0].axis('off')

axes[1].imshow(mask, cmap='tab10')
axes[1].set_title('Máscara (todas as classes)')
axes[1].axis('off')

# Isola cada valor para comparar visualmente com a imagem
# e identificar qual corresponde à água
for val in sorted(np.unique(mask)):
    regiao = (mask == val)
    print(f'Valor {val:3d}: {regiao.sum():8,} pixels ({regiao.mean()*100:.1f}%)')

axes[2].imshow((mask == 29).astype(np.uint8), cmap='Blues')
axes[2].set_title('Classe 29 isolada (água)')
axes[2].axis('off')

plt.tight_layout()
plt.show()


## 2. Primeira tentativa — ResNet34 local

A primeira abordagem foi rodar o modelo localmente com uma GPU mais modesta (MX570). Algumas decisões tomadas nessa fase que depois se mostraram erradas:

- **Valor errado para água:** a máscara foi binarizada usando o intervalo `(mask > 20) & (mask < 40)` — uma heurística imprecisa que capturava pixels de outras classes junto com a água
- **Resolução baixa:** imagens redimensionadas para 256×256 para caber na VRAM, mas isso prejudicava rios finos
- **Arquitetura:** DeepLabV3+ com EfficientNet-b3, que gerava OOM (out of memory) frequente na GPU local


In [ ]:
# TENTATIVA 1 — DeepLabV3+ local (DESCARTADA)
# Problema principal: valor de máscara incorreto e OOM na GPU local

import torch
import torch.nn as nn
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMG_SIZE = (256, 256)  # resolução reduzida por limitação de VRAM

IMG_FOLDERS = [
    r'data/Train/GID-img-1',
    r'data/Train/GID-img-2',
    r'data/Train/GID-img-3',
    r'data/Train/GID-img-4',
]
LBL_DIR = r'data/Train/GID-label'

train_transform = A.Compose([
    A.RandomResizedCrop(IMG_SIZE[0], IMG_SIZE[1], scale=(0.7, 1.0), p=0.5),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ColorJitter(brightness=0.2, contrast=0.2, p=0.3),
    A.GaussNoise(p=0.2),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

class GIDDataset(Dataset):
    def __init__(self, img_folders, lbl_dir, transform=None):
        self.pairs = []
        for folder in img_folders:
            if os.path.exists(folder):
                for f in os.listdir(folder):
                    lbl_p = os.path.join(lbl_dir, os.path.splitext(f)[0] + '.png')
                    if os.path.exists(lbl_p):
                        self.pairs.append((os.path.join(folder, f), lbl_p))
        self.transform = transform

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, lbl_path = self.pairs[idx]
        image     = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
        mask_gray = cv2.imread(lbl_path, cv2.IMREAD_GRAYSCALE)
        # ERRO: intervalo impreciso capturava classes vizinhas (valores 20-40)
        # o valor correto descoberto depois foi exatamente 29
        mask = ((mask_gray > 20) & (mask_gray < 40)).astype(np.float32)
        if self.transform:
            aug = self.transform(image=image, mask=mask)
            image, mask = aug['image'], aug['mask'].unsqueeze(0)
        return image, mask

# DeepLabV3+ com EfficientNet-b3 — gerava OOM na GPU local com batch > 2
model_v1 = smp.DeepLabV3Plus(
    encoder_name='efficientnet-b3',
    encoder_weights='imagenet',
    in_channels=3,
    classes=1
).to(DEVICE)

dice_loss  = smp.losses.DiceLoss(mode='binary')
focal_loss = smp.losses.FocalLoss(mode='binary')

def criterion_v1(y_pred, y_true):
    return 0.5 * dice_loss(y_pred, y_true) + 0.5 * focal_loss(y_pred, y_true)

print('Arquitetura v1 carregada. Essa versão foi descartada por OOM e máscara incorreta.')
print(f'Parâmetros do encoder: {sum(p.numel() for p in model_v1.encoder.parameters()):,}')


## 3. Segunda tentativa — migração para o Kaggle

A GPU P100 disponível gratuitamente no Kaggle tem 16GB de VRAM, o que abriu espaço para:
- Aumentar a resolução para 512×512
- Usar gradient accumulation para simular batches maiores
- Treinar por mais épocas com checkpoint para retomar sessões

Nessa fase o valor da máscara foi corrigido para `== 2` (outra tentativa equivocada antes de chegar no valor correto `== 29`), e a arquitetura mudou para UNet++ com atenção espacial (scSE).


In [ ]:
# TENTATIVA 2 — UNet++ no Kaggle com valor de máscara ainda incorreto
# mask == 2 era errado; o valor correto confirmado visualmente foi 29

# Trecho do dataset com o erro de valor de máscara
class WaterDataset_v2(Dataset):
    def __init__(self, pairs, transform=None):
        self.pairs = pairs
        self.transform = transform

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_p, lbl_p = self.pairs[idx]
        image    = cv2.cvtColor(cv2.imread(img_p), cv2.COLOR_BGR2RGB)
        mask_raw = cv2.imread(lbl_p, cv2.IMREAD_GRAYSCALE)
        # AINDA ERRADO: == 2 não correspondia à água neste dataset
        mask = (mask_raw == 2).astype(np.float32)
        if self.transform:
            aug = self.transform(image=image, mask=mask)
            image, mask = aug['image'], aug['mask'].unsqueeze(0)
        return image, mask

# UNet++ com scSE — atenção espacial e de canal simultaneamente
# Ajuda o modelo a focar nas regiões com água, que são minoria
model_v2 = smp.UnetPlusPlus(
    encoder_name='efficientnet-b3',
    encoder_weights='imagenet',
    decoder_attention_type='scse',
    in_channels=3,
    classes=1,
    activation=None,
).to(DEVICE)

print('Arquitetura v2 (UNet++ + scSE) carregada.')
print('Problema desta versão: máscara binarizada com valor errado (== 2 em vez de == 29)')


## 4. O desbalanceamento de classes

Um problema central do dataset é que a água ocupa uma fração pequena da maioria das imagens — rios finos, lagos parcialmente visíveis. Modelos treinados sem cuidado tendem a prever tudo como "sem água" e ainda assim obter boa acurácia pixel a pixel.

A métrica da competição era o **Kappa de Cohen**, que penaliza esse comportamento porque leva em conta o acerto esperado por acaso. Para lidar com o desbalanceamento foram testadas diferentes funções de perda ao longo das tentativas.


In [ ]:
# Estimativa do desbalanceamento real no dataset
# Quanto mais alto o ratio, mais difícil é para o modelo aprender a classe minoritária

label_files_all = glob(os.path.join(LBL_DIR, '*.png'))
total_agua, total_resto = 0, 0

for f in label_files_all[:100]:  # amostra de 100 imagens
    m = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
    agua  = (m == 29).sum()
    resto = m.size - agua
    total_agua  += agua
    total_resto += resto

ratio = total_resto / (total_agua + 1e-7)
print(f'Pixels de água  : {total_agua:,}')
print(f'Pixels sem água : {total_resto:,}')
print(f'Ratio (neg/pos) : {ratio:.1f}x')
print()
print('Isso significa que para cada pixel de água há ~{:.0f} pixels de outras classes.'.format(ratio))
print('Funções de perda como BCE padrão tendem a ignorar a classe minoritária nesse cenário.')


## 5. Progressão das funções de perda testadas

| Versão | Loss | Problema |
|---|---|---|
| v1 | Dice + Focal (50/50) | Mask errada, não deu para avaliar |
| v2 | Dice + BCE (80/20) | Mask errada, não deu para avaliar |
| v3 (final) | FocalTversky + BCE + Lovász | Mask correta, melhor resultado |

A **FocalTversky Loss** com `alpha=0.7, beta=0.3` penaliza mais os falsos negativos (água não detectada) do que os falsos positivos, o que faz sentido para um problema onde a classe é rara e cara de perder. O expoente `gamma=1.5` foca o gradiente nos exemplos mais difíceis.

A **Lovász Loss** é uma aproximação diferenciável do IoU e tende a melhorar a qualidade das bordas das máscaras preditas.
